# Supervisor calling the deployed Price Intelligence agent

Same as `supervisor.ipynb`, but the price intelligence sub-agent runs **remotely**: the tool calls the `lseg-price-intelligence-agent` deployment in LangSmith over HTTP with `RemoteGraph`, instead of importing it and running it in this notebook.

| | `supervisor.ipynb` + `price_intelligence` import | this notebook |
|---|---|---|
| Price agent code runs | in this notebook (local) | in the LangSmith deployment |
| Code changes take effect | immediately | after `langgraph deploy` |
| Traces | one trace | supervisor trace here, price agent trace in the deployment's project |

In [1]:
# Let Python find agents/ (same as supervisor.ipynb), and load the repo-root .env
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path().resolve().parent  # agents/
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

_ = load_dotenv(project_root.parent / ".env", override=True)  # finsights/.env (models.py looks in agents/)

In [2]:
import os

from langchain.agents import create_agent
from langchain.tools import tool
from langgraph.pregel.remote import RemoteGraph

from models import model
from utils.state import InvestigationContext, get_company, get_date

In [3]:
# The deployed agent. Pass the LangSmith API key explicitly: RemoteGraph doesn't send it otherwise.
PRICE_AGENT_URL = "https://lseg-price-intelligence-age-c6f431513939515ca815b94e6c2d7072.us.langgraph.app"

remote_price_agent = RemoteGraph(
    "lseg_price_intelligence_agent", url=PRICE_AGENT_URL, api_key=os.environ["LANGSMITH_API_KEY"]
)


@tool(parse_docstring=True)
def price_intelligence(company_or_ticker: str) -> dict | str:
    """Price Intelligence for a company: latest price, return, high and low, volume, benchmark comparison,
    VIX, volatility regime, trend, risk flags, key moves and a short summary, from historical data.

    Args:
        company_or_ticker: A company name such as "NVIDIA" or a ticker such as "NVDA".
    """
    result = remote_price_agent.invoke(
        {"messages": [{"role": "user", "content": f"Analyse the price of {company_or_ticker}."}]}
    )
    report = result.get("structured_response")
    if report and "latest_price" in report:  # a full PriceIntelligenceReport
        return report
    last = result["messages"][-1]
    return last["content"] if isinstance(last, dict) else last.content  # e.g. no data for that company

In [4]:
# Quick check: call the deployed agent directly
report = price_intelligence.invoke({"company_or_ticker": "NVIDIA"})
{k: report[k] for k in ["ticker", "as_of", "data_status", "latest_price", "period_return_pct", "relative_performance_pct", "vix_latest", "regime", "trend"]}

{'ticker': 'NVDA',
 'as_of': '2026-11-20T21:00:00Z',
 'data_status': 'historical',
 'latest_price': 192.93,
 'period_return_pct': 3.5,
 'relative_performance_pct': 4.18,
 'vix_latest': 18.9,
 'regime': 'elevated',
 'trend': 'volatility_rising'}

In [5]:
agent = create_agent(
    model=model,
    context_schema=InvestigationContext,
    tools=[get_company, get_date, price_intelligence],
)

In [6]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "How has the company's share price behaved this week?"}]},
    context=InvestigationContext(company_name="NVIDIA", date_str="2026-11-20"),
)

for message in response["messages"]:
    for call in getattr(message, "tool_calls", None) or []:
        print("supervisor called:", call["name"], call["args"])

print()
print(response["messages"][-1].text)

supervisor called: get_company {}
supervisor called: get_date {}
supervisor called: price_intelligence {'company_or_ticker': 'NVIDIA'}

NVIDIA rose **3.5% this week**, ending Friday, Nov. 20, at **$192.93**. It was a volatile ride: the shares ranged from **$182.90** to **$200.29**, and Thursday saw a sharp gap up followed by an intraday reversal.

Despite the swings, NVIDIA outperformed the **QQQ**, which fell **0.68%** over the same period. Trading volume was about **24% above average**.
